# Módulo 4 · Clase 8 (Práctica) — Generative AI, multimodal y agentes en código
### Deep Learning · Laboratorio · Cierre del curso

> **Para el ayudante:** laboratorio de 3 horas para cerrar el curso con las herramientas de hoy. **Activa GPU.** Patrón: explico → corremos → resuelven un `TODO` (soluciones en celdas `# @title Solución`).

**Objetivos.** Cada estudiante habrá:
1. Usado **CLIP** para clasificación zero-shot y búsqueda imagen-texto.
2. Generado imágenes con un modelo de **difusión** (Stable Diffusion).
3. Construido un sistema **RAG** (recuperación + generación).
4. Visto el esqueleto de un **agente** con uso de herramientas.

**Agenda (≈ 3 h):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Setup | 10 |
| 1 | CLIP: zero-shot y búsqueda imagen-texto | 40 |
| 2 | Generación de imágenes con difusión | 40 |
| — | *Descanso* | 10 |
| 3 | RAG: pregunta-respuesta sobre documentos | 45 |
| 4 | Un agente con herramientas (bonus) | 25 |
| 5 | Cierre | 10 |


In [ ]:
# Bloque 0 · Setup
!pip install -q diffusers transformers accelerate sentence-transformers
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("⚠️  La generación con difusión necesita GPU. Activa GPU en Colab.")

## Bloque 1 · CLIP: clasificación zero-shot

CLIP alinea imágenes y texto en un espacio común. Para clasificar **sin entrenar**, comparamos el embedding de una imagen con los de varias frases candidatas y elegimos la más cercana.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import requests

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Una imagen de ejemplo (dos gatos en un sofá, del dataset COCO)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
imagen = Image.open(requests.get(url, stream=True).raw)
imagen

In [ ]:
# Clasificación zero-shot: comparamos la imagen con frases candidatas
etiquetas = ["una foto de gatos", "una foto de un perro", "una foto de un auto", "una foto de comida"]
inputs = proc(text=etiquetas, images=imagen, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    logits = clip(**inputs).logits_per_image          # similitud imagen-texto
    probs = logits.softmax(dim=1)[0]
for etq, p in sorted(zip(etiquetas, probs.tolist()), key=lambda x:-x[1]):
    print(f"{p:6.1%}  {etq}")

### 🧩 Ejercicio 1 — Búsqueda imagen→texto
Dada una lista de descripciones tuyas, encuentra cuál describe mejor la imagen. Luego prueba con tus propias frases y observa cómo CLIP entiende conceptos que nunca vio explícitamente etiquetados.

In [ ]:
# @title Solución
def clasificar(imagen, etiquetas):
    inputs = proc(text=etiquetas, images=imagen, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        probs = clip(**inputs).logits_per_image.softmax(dim=1)[0]
    return sorted(zip(etiquetas, probs.tolist()), key=lambda x:-x[1])

mis_etiquetas = ["dos gatos durmiendo", "un gato jugando", "un paisaje de montaña", "personas en una fiesta"]
for etq, p in clasificar(imagen, mis_etiquetas):
    print(f"{p:6.1%}  {etq}")

## Bloque 2 · Generación de imágenes con difusión

Usamos **Stable Diffusion** vía la librería `diffusers`. A partir de un *prompt* de texto, el modelo parte de ruido y lo va limpiando, condicionado por el texto, hasta producir una imagen.

> **Nota de cómputo:** descarga ~4 GB y requiere GPU. Usamos `torch.float16` y pocos pasos para que sea rápido. Si va lento, reduce `num_inference_steps`.

In [ ]:
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,
).to(device)
pipe.set_progress_bar_config(disable=True)

prompt = "a watercolor painting of a fox reading a book in a cozy library, warm light"
imagen_gen = pipe(prompt, num_inference_steps=25, guidance_scale=7.5).images[0]
imagen_gen

El parámetro `guidance_scale` controla cuánto se "obliga" a seguir el prompt (más alto = más fiel al texto, menos diverso). `num_inference_steps` controla cuántos pasos de *denoising* se aplican (más = mejor calidad, más lento).

### 🧩 Ejercicio 2
Genera variaciones cambiando el prompt y el `guidance_scale`. Fija una `seed` con un generador para reproducibilidad y compara dos valores de guía.

In [ ]:
# @title Solución
def generar(prompt, scale=7.5, seed=0, steps=25):
    g = torch.Generator(device=device).manual_seed(seed)
    return pipe(prompt, num_inference_steps=steps, guidance_scale=scale, generator=g).images[0]

prompt = "a futuristic city at sunset, synthwave style, highly detailed"
img_a = generar(prompt, scale=3.0, seed=42)
img_b = generar(prompt, scale=12.0, seed=42)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(9,5))
ax[0].imshow(img_a); ax[0].set_title("guidance=3 (más libre)"); ax[0].axis('off')
ax[1].imshow(img_b); ax[1].set_title("guidance=12 (más fiel)"); ax[1].axis('off')
plt.show()

## Bloque 3 · RAG: preguntas y respuestas sobre documentos

Construimos un RAG real: embeddings con **sentence-transformers**, recuperación por similitud, y generación con un LLM pequeño (**Flan-T5**). Como corpus usamos notas del propio curso.

In [ ]:
from sentence_transformers import SentenceTransformer, util
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# "Base de conocimiento": notas del curso (en un caso real serían cientos de documentos)
corpus = [
    "El perceptrón multicapa (MLP) se entrena con backpropagation y gradiente descendente.",
    "Las redes convolucionales (CNN) usan filtros que explotan la estructura espacial de las imágenes.",
    "ResNet introdujo las conexiones residuales para entrenar redes muy profundas.",
    "El transfer learning reutiliza un modelo preentrenado y lo adapta a una nueva tarea.",
    "Los Transformers reemplazan la recurrencia por el mecanismo de self-attention.",
    "BERT es un Transformer solo-encoder preentrenado con masked language modeling.",
    "Stable Diffusion genera imágenes con un proceso de difusión en un espacio latente.",
    "CLIP alinea imágenes y texto en un espacio común mediante aprendizaje contrastivo.",
]
corpus_emb = encoder.encode(corpus, convert_to_tensor=True)
print("corpus indexado:", corpus_emb.shape)

In [ ]:
# Recuperación: dada una pregunta, buscar los chunks más relevantes
def recuperar(pregunta, k=2):
    q = encoder.encode(pregunta, convert_to_tensor=True)
    hits = util.cos_sim(q, corpus_emb)[0]
    top = torch.topk(hits, k).indices.tolist()
    return [corpus[i] for i in top]

pregunta = "¿Cómo se entrenan las redes profundas muy grandes?"
contexto = recuperar(pregunta)
print("Pregunta:", pregunta)
print("\nChunks recuperados:")
for c in contexto: print(" -", c)

In [ ]:
# Generación: el LLM responde usando SOLO el contexto recuperado
from transformers import pipeline
llm = pipeline("text2text-generation", model="google/flan-t5-base",
               device=0 if device=="cuda" else -1)

def responder(pregunta, k=2):
    contexto = recuperar(pregunta, k)
    prompt = ("Responde la pregunta usando solo el contexto.\n"
              f"Contexto: {' '.join(contexto)}\n"
              f"Pregunta: {pregunta}\nRespuesta:")
    return llm(prompt, max_new_tokens=60)[0]["generated_text"]

for q in ["¿Cómo se entrenan las redes profundas muy grandes?",
          "¿Qué hace CLIP?",
          "¿Cómo genera imágenes Stable Diffusion?"]:
    print("P:", q); print("R:", responder(q), "\n")

Esto es la esencia de un asistente sobre documentos propios (manuales, apuntes, papers): el LLM responde **fundamentado** en tu base de conocimiento, reduciendo alucinaciones.

### 🧩 Ejercicio 3
Agrega tus propios documentos al `corpus`, re-indexa y haz preguntas. ¿Qué pasa si preguntas algo que **no** está en el corpus? Observa cómo responde y comenta por qué es importante el paso de recuperación.

In [ ]:
# @title Solución
corpus.append("El dropout apaga neuronas al azar durante el entrenamiento para reducir el overfitting.")
corpus_emb = encoder.encode(corpus, convert_to_tensor=True)   # re-indexar
print("Pregunta en el corpus:")
print("R:", responder("¿Para qué sirve el dropout?"))
print("\nPregunta fuera del corpus:")
print("R:", responder("¿Cuál es la capital de Francia?"))
print("\n(El modelo intenta responder con el contexto recuperado, aunque no sea relevante:")
print(" en un sistema real se filtra por umbral de similitud o se responde 'no lo sé'.)")

## Bloque 4 · Un agente con herramientas (bonus)

Cerramos con el patrón **ReAct**: un LLM que razona y llama **herramientas**. Implementamos un agente mínimo con dos herramientas (una calculadora y nuestro buscador del corpus) y un orquestador en Python.

> Los modelos pequeños razonan de forma limitada; aquí el objetivo es ver la **mecánica** del bucle. Con un LLM potente (GPT-4, Claude, Llama grande) este mismo patrón resuelve tareas complejas de forma fiable.

In [ ]:
import re
# Herramientas disponibles para el agente
def calculadora(expr): return str(eval(expr, {"__builtins__": {}}))
def buscar(consulta):  return recuperar(consulta, k=1)[0]
HERRAMIENTAS = {"calculadora": calculadora, "buscar": buscar}

def ejecutar_accion(texto_accion):
    # parsea  nombre('argumento')  y llama la herramienta
    m = re.match(r"(\w+)\('(.+)'\)", texto_accion.strip())
    if not m: return "acción no reconocida"
    nombre, arg = m.groups()
    return HERRAMIENTAS.get(nombre, lambda x:"herramienta desconocida")(arg)

# Demo del bucle (acciones predefinidas para ilustrar; un LLM las generaría)
print("Tarea: ¿Cuánto es 12*8 y qué es self-attention?\n")
for paso in ["calculadora('12*8')", "buscar('self-attention en Transformers')"]:
    print(f"Action     : {paso}")
    print(f"Observation: {ejecutar_accion(paso)}\n")

### 🧩 Ejercicio 4
Agrega una tercera herramienta (por ejemplo, `longitud('texto')` que devuelva el número de palabras) y úsala. Piensa: ¿cómo decidiría un LLM *qué* herramienta usar y *cuándo* parar? (Pista: el prompt de ReAct le enseña el formato Thought/Action/Observation con ejemplos.)

In [ ]:
# TODO: agrega tu herramienta y pruébala
# HERRAMIENTAS["longitud"] = lambda t: ...
...

## Bloque 5 · Cierre del curso

¡Felicitaciones! En este laboratorio final usaste las herramientas que definen la IA de hoy: modelos multimodales (CLIP), generación por difusión y sistemas RAG con agentes.

A lo largo de los 4 módulos pasaste de implementar un MLP desde cero a orquestar modelos generativos y agentes. La constante: **datos + modelo + pérdida + optimización**. Con esa base, estás en condiciones de entender y construir sobre lo que venga.

**Para seguir:** los labs adicionales del repositorio (`Labs/`), los papers originales de cada tema, y proyectos propios. La mejor forma de aprender deep learning es construir.